In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Load the dataset
df = pd.read_csv("robot_braking_data.csv")



# 2. Select the target column
# CHANGE this to the actual column you want to predict
TARGET_COLUMN = "stopping_distance_m"

# 3. Separate input features and target
df = df.dropna(subset=[TARGET_COLUMN])

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

# This version expects numerical input features and target
X = X.select_dtypes(include=[np.number])

# Remove rows with missing numerical input values
valid_rows = X.notna().all(axis=1)
X = X.loc[valid_rows]
y = y.loc[valid_rows]

if X.shape[1] == 0:
    raise ValueError("No numerical input features were found.")

# 4. Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Total samples used:", len(X))

# 5. Create Ridge Regression model
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

# 6. Create Lasso Regression model
lasso_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=1.0, max_iter=100000))
])

# 7. Train both models
ridge_model.fit(X_train, y_train)
lasso_model.fit(X_train, y_train)

# 8. Predict on unseen test data
y_pred_ridge = ridge_model.predict(X_test)
y_pred_lasso = lasso_model.predict(X_test)

# 9. Calculate performance metrics
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R_squared": r2
    }

ridge_results = evaluate_model(
    y_test, y_pred_ridge, "Ridge Regression"
)

lasso_results = evaluate_model(
    y_test, y_pred_lasso, "Lasso Regression"
)

# 10. Compare the results
results = pd.DataFrame([
    ridge_results,
    lasso_results
])

print("\nModel Performance on Test Data:")
print(results.to_string(index=False, float_format="%.4f"))

# 11. Save predictions and actual values
predictions = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Ridge_Predicted": y_pred_ridge,
    "Lasso_Predicted": y_pred_lasso
})

predictions.to_csv("robot_braking_predictions.csv", index=False)

print("\nPredictions saved to robot_braking_predictions.csv")


Dataset shape: (40, 2)

Column names:
['speed_mps', 'stopping_distance_m']

First 5 rows:
   speed_mps  stopping_distance_m
0       0.21                 0.02
1       0.22                 0.04
2       0.24                 0.04
3       0.34                 0.02
4       0.37                 0.07

Training samples: 32
Testing samples: 8
Total samples used: 40

Model Performance on Test Data:
           Model    MAE    MSE   RMSE  R_squared
Ridge Regression 0.3139 0.1054 0.3246     0.9346
Lasso Regression 0.7605 0.8582 0.9264     0.4669

Predictions saved to robot_braking_predictions.csv
